In [5]:
print('Đang khai báo thư viện')

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os

print('Khai báo thư viện thành công')

Đang khai báo thư viện
Khai báo thư viện thành công


In [6]:
print('Khai báo dữ liệu')

file_path = os.path.join("..", "Notebooks", "DataPreprocessing", "20260521_221410_data_after_categorial.csv")

df = pd.read_csv(file_path)

print("=" * 60)
print("DỮ LIỆU SAU TIỀN XỬ LÝ")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"\nCác cột hiện có:\n{df.columns.tolist()}")
print(f"\nClass distribution:\n{df['Class'].value_counts().sort_index()}")

Khai báo dữ liệu
DỮ LIỆU SAU TIỀN XỬ LÝ
Shape: (11516, 20)

Các cột hiện có:
['Id', 'Artist Name', 'Track Name', 'Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature', 'Class', 'key_sin', 'key_cos']

Class distribution:
Class
0      400
1      878
2      814
3      258
4      248
5      926
6     1655
7      369
8     1186
9     1615
10    3167
Name: count, dtype: int64


In [ ]:
# ============================================
# 2. TÁCH X VÀ y
# ============================================
# Bỏ các cột không cần thiết

drop_cols = ['Id', 'Artist Name', 'Track Name']
existing_drop = [col for col in drop_cols if col in df.columns]
X = df.drop(columns=['Class'] + existing_drop)
y = df['Class']

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:




# ============================================
# 3. CHIA TRAIN / VALIDATION
# ============================================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Giữ tỷ lệ class giống nhau
)

print(f"\nTrain size: {X_train.shape[0]}")
print(f"Validation size: {X_val.shape[0]}")
print(f"\nTrain class distribution:\n{y_train.value_counts().sort_index()}")

# ============================================
# 4. TRAIN RANDOM FOREST (phiên bản cơ bản)
# ============================================
print("\n" + "=" * 60)
print("RANDOM FOREST - BASIC")
print("=" * 60)

rf_basic = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',  # Tự động cân bằng class
    n_jobs=-1  # Dùng tất cả CPU core
)

rf_basic.fit(X_train, y_train)
y_pred_basic = rf_basic.predict(X_val)

# Đánh giá
print(f"\nAccuracy: {accuracy_score(y_val, y_pred_basic):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred_basic, zero_division=0))

# ============================================
# 5. RANDOM FOREST - TUNING HYPERPARAMETERS
# ============================================
print("\n" + "=" * 60)
print("RANDOM FOREST - TUNING")
print("=" * 60)

# Định nghĩa các tham số cần tìm
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# GridSearchCV (chạy hơi lâu nếu data to)
# Nếu muốn nhanh hơn, dùng RandomizedSearchCV
from sklearn.model_selection import RandomizedSearchCV

rf_tuned = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    rf_tuned,
    param_distributions=param_grid,
    n_iter=20,  # Thử 20 bộ tham số
    cv=5,       # 5-fold cross validation
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Đang tìm tham số tối ưu... (có thể hơi lâu)")
random_search.fit(X_train, y_train)

print(f"\nBest parameters: {random_search.best_params_}")
print(f"Best cross-validation score: {random_search.best_score_:.4f}")

# Dùng model tốt nhất để predict
rf_best = random_search.best_estimator_
y_pred_tuned = rf_best.predict(X_val)

print(f"\nValidation Accuracy sau tuning: {accuracy_score(y_val, y_pred_tuned):.4f}")

# ============================================
# 6. SO SÁNH KẾT QUẢ
# ============================================
print("\n" + "=" * 60)
print("SO SÁNH KẾT QUẢ")
print("=" * 60)

print("\n--- RANDOM FOREST BASIC ---")
print(f"Accuracy: {accuracy_score(y_val, y_pred_basic):.4f}")
print(classification_report(y_val, y_pred_basic, zero_division=0))

print("\n--- RANDOM FOREST TUNED ---")
print(f"Accuracy: {accuracy_score(y_val, y_pred_tuned):.4f}")
print(classification_report(y_val, y_pred_tuned, zero_division=0))

# ============================================
# 7. CONFUSION MATRIX
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Confusion matrix cho basic model
cm_basic = confusion_matrix(y_val, y_pred_basic)
sns.heatmap(cm_basic, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=range(11), yticklabels=range(11))
axes[0].set_title('Random Forest Basic - Confusion Matrix', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Confusion matrix cho tuned model
cm_tuned = confusion_matrix(y_val, y_pred_tuned)
sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=range(11), yticklabels=range(11))
axes[1].set_title('Random Forest Tuned - Confusion Matrix', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.suptitle('Comparison of Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ============================================
# 8. FEATURE IMPORTANCE
# ============================================
feat_imp = pd.Series(rf_best.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(feat_imp)))
feat_imp.plot(kind='bar', color=colors, edgecolor='black')
plt.title('Feature Importance - Random Forest (Tuned)', fontweight='bold', fontsize=14)
plt.ylabel('Importance Score', fontsize=12)
plt.xlabel('Features', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.tight_layout()
plt.show()

# In top 10 features
print("\n" + "=" * 60)
print("TOP 10 IMPORTANT FEATURES")
print("=" * 60)
for i, (feat, imp) in enumerate(feat_imp.head(10).items(), 1):
    print(f"{i:2d}. {feat:20s} : {imp:.4f}")

# ============================================
# 9. CROSS-VALIDATION SCORE (để đánh giá tổng quát hơn)
# ============================================
print("\n" + "=" * 60)
print("CROSS-VALIDATION SCORE")
print("=" * 60)

cv_scores = cross_val_score(rf_best, X, y, cv=5, scoring='accuracy')
print(f"5-fold CV scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# ============================================
# 10. LƯU MODEL (nếu muốn dùng sau)
# ============================================
import joblib

# Lưu model
joblib.dump(rf_best, 'random_forest_best_model.pkl')
print("\n✅ Đã lưu model vào file: random_forest_best_model.pkl")

# ============================================
# 11. DỰ ĐOÁN BÀI HÁT MỚI (ví dụ)
# ============================================
print("\n" + "=" * 60)
print("VÍ DỤ DỰ ĐOÁN BÀI HÁT MỚI")
print("=" * 60)

# Lấy 1 bài từ validation set để test thử
sample = X_val.iloc[0:1]
true_class = y_val.iloc[0]
pred_class = rf_best.predict(sample)[0]

print(f"True class: {true_class}")
print(f"Predicted class: {pred_class}")
print(f"✅ Dự đoán {'đúng' if true_class == pred_class else 'sai'}")